# 🔬 PTSD Wafer Probing Simulation
**Based on:** *Machine-Learning Driven Sensor Data Analytics for Yield Enhancement of Wafer Probing*  
IEEE International Test Conference (ITC) 2023 — DOI: 10.1109/ITC51656.2023.00023

---
### Framework Phases:
- **Phase I** — Data Generation & Preprocessing
- **Phase II** — Feature Extraction & Selection (Statistical, Frequency, Wavelet, PCA)
- **Phase III** — Model Training (SVM, Decision Tree, Random Forest)
- **Phase IV** — Evaluation & Visualization
---

## ⚙️ Step 0 — Install & Import Libraries

In [ ]:
# Install dependencies (already available in Colab)
!pip install -q numpy pandas scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.utils import resample

import os
os.makedirs('outputs', exist_ok=True)

np.random.seed(42)
print('✅ Libraries loaded successfully!')

## ⚙️ Step 1 — Configuration (Edit These Parameters)

In [ ]:
# ============================================================
# CONFIGURATION — Edit these to change the simulation
# ============================================================
N_TOTAL       = 9200     # Total observations (paper: >9000)
N_PRODUCTS    = 5        # Number of products
FAILURE_RATE  = 0.01     # ~1% failure (paper: >99% pass)
RANDOM_SEED   = 42
TRAIN_RATIO   = 0.70     # Paper: 70% train
VAL_RATIO     = 0.10     # Paper: 10% validation
TEST_RATIO    = 0.20     # Paper: 20% test

np.random.seed(RANDOM_SEED)
print(f'Config: {N_TOTAL} samples | {N_PRODUCTS} products | {FAILURE_RATE*100:.1f}% failure rate')

## 📡 Phase I — Data Generation & Preprocessing

In [ ]:
# ============================================================
# GENERATE SYNTHETIC SENSOR DATA
# Mirrors NXP UF3000EX prober variables from the PTSD paper
# ============================================================

N = N_TOTAL
products = [f'Product_{i+1}' for i in range(N_PRODUCTS)]
product_ids = np.random.choice(products, size=N, p=[0.25, 0.20, 0.20, 0.20, 0.15])

# Touchdowns — drives contamination buildup
touchdowns = np.random.randint(500, 50000, size=N)

# CRES: Contact Resistance (Ohm) — spikes when contaminated (>30k touchdowns)
base_cres    = np.random.normal(0.50, 0.10, N)
debris_spike = np.random.exponential(0.50, N) * (touchdowns > 30000)
cres         = np.clip(base_cres + debris_spike, 0.05, 5.0)

# zHeight: Probe-to-wafer distance (microns)
zHeight = np.clip(np.random.normal(125, 10, N), 90, 180)

# Over-travel (microns)
over_travel = np.clip(np.random.normal(75, 10, N), 30, 150)

# Planarity (microns) — lower is better
planarity = np.clip(np.random.exponential(3, N) + 1, 0.5, 30)

# Tip Diameter (microns) — degrades with touchdowns
tip_diam = np.clip(np.random.normal(30, 2, N) - (touchdowns / 100000) * 5, 15, 40)

# Applied Force Fc (grams)
fc = np.clip(np.random.normal(3.5, 0.4, N), 2.0, 6.0)

# Temperature (Celsius)
temperature = np.random.normal(25, 3, N)

# Site Number (multi-site probing, 1-16)
site_number = np.random.randint(1, 17, N)

# ---- Labels: Failure conditions from paper ----
fail_cres      = cres > 1.5
fail_planarity = planarity > 15
fail_tip       = tip_diam < 20
needle_fail    = fail_cres | fail_planarity | fail_tip

# Ensure target failure rate
current_rate = needle_fail.mean()
if current_rate < FAILURE_RATE:
    extra = np.random.random(N) < (FAILURE_RATE - current_rate)
    needle_fail = needle_fail | (extra & ~needle_fail)

needle_status = np.where(needle_fail, 'FAIL', 'PASS')

# Defect patterns
defect_pattern = np.where(needle_fail,
    np.where(fail_cres, 'Bonded_Debris',
    np.where(fail_planarity, 'Uneven_Wear', 'Tip_Degradation')), 'None')

# Bin numbers (1=best, 5=worst)
bin_number = np.ones(N, dtype=int)
bin_number[cres > 0.8] = 2
bin_number[cres > 1.2] = 3
bin_number[needle_fail] = np.random.choice([4, 5], size=needle_fail.sum())

# Build DataFrame
df = pd.DataFrame({
    'Wafer_ID':          [f'W{np.random.randint(1000,9999)}' for _ in range(N)],
    'Product':           product_ids,
    'Site_Number':       site_number,
    'Touchdowns':        touchdowns,
    'CRES_Ohm':          np.round(cres, 4),
    'zHeight_um':        np.round(zHeight, 2),
    'OverTravel_um':     np.round(over_travel, 2),
    'Planarity_um':      np.round(planarity, 2),
    'TipDiameter_um':    np.round(tip_diam, 2),
    'AppliedForce_Fc_g': np.round(fc, 3),
    'Temperature_C':     np.round(temperature, 1),
    'Bin_Number':        bin_number,
    'Needle_Status':     needle_status,
    'Defect_Pattern':    defect_pattern
})

print(f'✅ Dataset generated: {len(df)} rows')
print(f'   PASS: {(df.Needle_Status=="PASS").sum()} | FAIL: {(df.Needle_Status=="FAIL").sum()} ({needle_fail.mean()*100:.2f}% failure rate)')
df.head()

In [ ]:
# ---- Export CSV ----
df.to_csv('outputs/wafer_probing_dataset.csv', index=False)
print('✅ CSV exported to outputs/wafer_probing_dataset.csv')
df.describe()

## 📊 Phase I (cont.) — Data Visualization

In [ ]:
# ============================================================
# FIGURE 1: CRES vs Touchdowns (Paper Fig.2 equivalent)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))

pass_mask = needle_fail == False
ax.scatter(touchdowns[pass_mask], cres[pass_mask], s=8, alpha=0.3,
           color='steelblue', label='PASS')
ax.scatter(touchdowns[needle_fail], cres[needle_fail], s=30, alpha=0.7,
           color='crimson', marker='^', label='FAIL')
ax.axhline(1.5, color='red', linestyle='--', linewidth=1.5, label='Failure Threshold (1.5Ω)')
ax.axvline(30000, color='black', linestyle='--', linewidth=1.2, label='30k Touchdowns')

ax.set_xlabel('Number of Touchdowns', fontsize=12)
ax.set_ylabel('Contact Resistance CRES (Ohm)', fontsize=12)
ax.set_title('CRES vs Touchdowns — Contamination Buildup', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/Fig1_CRES_vs_Touchdowns.png', dpi=150)
plt.show()
print('✅ Fig1 saved')

In [ ]:
# ============================================================
# FIGURE 2: Yield vs Touchdowns (Paper Fig.2 style)
# ============================================================
bins      = np.linspace(500, 50000, 50)
bin_yield = []
bin_centers = []
for i in range(len(bins)-1):
    mask = (touchdowns >= bins[i]) & (touchdowns < bins[i+1])
    if mask.sum() > 0:
        bin_yield.append((needle_fail[mask] == False).mean() * 100)
        bin_centers.append((bins[i] + bins[i+1]) / 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bin_centers, bin_yield, 'b-o', markersize=4, linewidth=2, label='Actual Wafer Yield')
ax.axhline(99, color='green', linestyle='--', linewidth=1.5, label='Max Yield')
ax.axhline(95, color='red', linestyle='--', linewidth=1.5, label='Min Yield Threshold')
ax.set_ylim([85, 101])
ax.set_xlabel('Number of Touchdowns', fontsize=12)
ax.set_ylabel('Yield (%)', fontsize=12)
ax.set_title('Yield Variation vs. Number of Touchdowns', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/Fig2_Yield_vs_Touchdowns.png', dpi=150)
plt.show()
print('✅ Fig2 saved')

In [ ]:
# ============================================================
# FIGURE 3: Feature Distributions — Pass vs Fail
# ============================================================
sensor_cols  = ['CRES_Ohm','zHeight_um','OverTravel_um','Planarity_um','TipDiameter_um','AppliedForce_Fc_g']
df_pass = df[df.Needle_Status == 'PASS']
df_fail = df[df.Needle_Status == 'FAIL']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), sensor_cols):
    ax.hist(df_pass[col], bins=40, alpha=0.6, color='steelblue',
            density=True, label='PASS')
    ax.hist(df_fail[col], bins=40, alpha=0.7, color='crimson',
            density=True, label='FAIL')
    ax.set_title(col.replace('_', ' '), fontsize=10, fontweight='bold')
    ax.set_xlabel('Value', fontsize=9); ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Sensor Feature Distributions: PASS vs FAIL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/Fig3_Feature_Distributions.png', dpi=150)
plt.show()
print('✅ Fig3 saved')

In [ ]:
# ============================================================
# FIGURE 4: Defect Patterns & Per-Product Failure Rate
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart — defect patterns
pattern_counts = df.Defect_Pattern.value_counts()
colors_pie = ['#d0d0d0','#e84040','#f09020','#3090e0']
ax1.pie(pattern_counts.values, labels=pattern_counts.index.str.replace('_',' '),
        autopct='%1.1f%%', colors=colors_pie, startangle=140,
        explode=[0.05]*len(pattern_counts))
ax1.set_title('Defect Pattern Distribution', fontsize=12, fontweight='bold')

# Bar chart — per product failure rate
fail_rates = df.groupby('Product').apply(lambda x: (x.Needle_Status=='FAIL').mean()*100)
ax2.bar(fail_rates.index, fail_rates.values, color='steelblue', edgecolor='navy')
ax2.set_xlabel('Product', fontsize=11); ax2.set_ylabel('Failure Rate (%)', fontsize=11)
ax2.set_title('Failure Rate per Product', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=20); ax2.grid(axis='y', alpha=0.3)

fig.suptitle('PTSD Framework — Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/Fig4_Defect_Product_Summary.png', dpi=150)
plt.show()
print('✅ Fig4 saved')

## 🔧 Phase II — Feature Extraction & PCA

In [ ]:
# ============================================================
# FEATURE ENGINEERING (Paper Sec. IV.B)
# Statistical + Frequency-domain + Wavelet-proxy + PCA
# ============================================================

feature_cols = ['CRES_Ohm','zHeight_um','OverTravel_um','Planarity_um',
                'TipDiameter_um','AppliedForce_Fc_g','Temperature_C',
                'Touchdowns','Site_Number','Bin_Number']
X_raw = df[feature_cols].values.astype(float)

# --- Statistical features: CRES mean/std by touchdown bin ---
td_bins  = np.linspace(500, 50000, 20)
cres_arr = df['CRES_Ohm'].values
td_arr   = df['Touchdowns'].values
cres_bin_mean = np.zeros(N)
cres_bin_std  = np.zeros(N)
for i in range(len(td_bins)-1):
    mask = (td_arr >= td_bins[i]) & (td_arr < td_bins[i+1])
    if mask.sum() > 1:
        cres_bin_mean[mask] = cres_arr[mask].mean()
        cres_bin_std[mask]  = cres_arr[mask].std()

# --- Wavelet proxy: IQR and range in rolling window ---
sort_idx   = np.argsort(td_arr)
window     = 50
cres_iqr   = np.zeros(N)
cres_range = np.zeros(N)
for i in range(N):
    lo = max(0, i - window//2)
    hi = min(N, i + window//2)
    win = cres_arr[sort_idx[lo:hi]]
    cres_iqr[sort_idx[i]]   = np.percentile(win,75) - np.percentile(win,25)
    cres_range[sort_idx[i]] = win.max() - win.min()

# --- Frequency proxy: CRES deviation from local mean ---
cres_hf_ratio       = np.abs(cres_arr - cres_bin_mean) / (cres_bin_std + 1e-6)
cres_planarity_ratio = cres_arr / (df['Planarity_um'].values + 0.1)

# --- Assemble full feature matrix ---
X = np.column_stack([X_raw, cres_bin_mean, cres_bin_std,
                     cres_iqr, cres_range, cres_hf_ratio, cres_planarity_ratio])

# --- Handle NaN ---
col_means = np.nanmean(X, axis=0)
for j in range(X.shape[1]):
    X[np.isnan(X[:,j]), j] = col_means[j]

# --- Standardize ---
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)

# --- PCA: retain 95% variance ---
pca = PCA(n_components=0.95, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_norm)

print(f'✅ Feature engineering complete')
print(f'   Raw features: {X.shape[1]} → PCA components: {X_pca.shape[1]} (95% variance retained)')

In [ ]:
# ============================================================
# TRAIN / VAL / TEST SPLIT + OVERSAMPLE (Paper Sec. IV.A)
# 70% train | 10% val | 20% test
# ============================================================
from sklearn.model_selection import train_test_split

y = needle_fail.astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_pca, y, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_RATIO/(TRAIN_RATIO+VAL_RATIO),
    random_state=RANDOM_SEED, stratify=y_temp)

# --- Oversample minority class (FAIL) with Gaussian noise augmentation ---
X_fail = X_train[y_train == 1]
X_pass = X_train[y_train == 0]
n_over  = len(X_pass) - len(X_fail)
X_syn   = X_fail[np.random.choice(len(X_fail), n_over)] + \
          np.random.normal(0, 0.02, (n_over, X_train.shape[1]))
X_train = np.vstack([X_train, X_syn])
y_train = np.concatenate([y_train, np.ones(n_over, dtype=int)])
shuffle = np.random.permutation(len(X_train))
X_train, y_train = X_train[shuffle], y_train[shuffle]

print(f'✅ Split complete')
print(f'   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'   Oversampled FAIL class: +{n_over} synthetic samples')

## 🤖 Phase III — Model Training (SVM, Decision Tree, Random Forest)

In [ ]:
# ============================================================
# TRAIN ALL 3 MODELS (Paper Section IV.C)
# ============================================================

models = {
    'SVM': SVC(kernel='rbf', C=10, class_weight={0:1, 1:10},
               gamma='scale', probability=True, random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=5,
                                             class_weight='balanced',
                                             random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15,
                                             class_weight='balanced',
                                             random_state=RANDOM_SEED, n_jobs=-1)
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    y_pred = mdl.predict(X_test)
    results[name] = {
        'model':     mdl,
        'y_pred':    y_pred,
        'accuracy':  accuracy_score(y_test, y_pred)  * 100,
        'precision': precision_score(y_test, y_pred, zero_division=0) * 100,
        'recall':    recall_score(y_test, y_pred, zero_division=0)    * 100,
        'f1':        f1_score(y_test, y_pred, zero_division=0)        * 100,
        'conf_mat':  confusion_matrix(y_test, y_pred)
    }
    print(f'✅ {name:15s} → Acc: {results[name]["accuracy"]:6.2f}% | '
          f'Prec: {results[name]["precision"]:6.2f}% | '
          f'Recall: {results[name]["recall"]:6.2f}% | '
          f'F1: {results[name]["f1"]:6.2f}%')

## 📈 Phase IV — Evaluation & Results

In [ ]:
# ============================================================
# FIGURE 5: Confusion Matrices
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = res['conf_mat']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred PASS','Pred FAIL'],
                yticklabels=['True PASS','True FAIL'],
                annot_kws={'size':14, 'weight':'bold'})
    ax.set_title(f"{name}\nAcc: {res['accuracy']:.1f}%", fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

fig.suptitle('Confusion Matrices — PTSD Framework Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/Fig5_Confusion_Matrices.png', dpi=150)
plt.show()
print('✅ Fig5 saved')

In [ ]:
# ============================================================
# FIGURE 6: Model Comparison Bar Chart
# ============================================================
metrics = ['accuracy','precision','recall','f1']
labels  = ['Accuracy','Precision','Recall','F1 Score']
x       = np.arange(len(results))
width   = 0.18
colors  = ['steelblue','darkorange','seagreen','mediumpurple']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (metric, label) in enumerate(zip(metrics, labels)):
    vals = [results[m][metric] for m in results]
    ax.bar(x + i*width, vals, width, label=label, color=colors[i], edgecolor='white')

ax.axhline(95, color='red', linestyle='--', linewidth=1.5, label='Paper Target (95%)')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(list(results.keys()), fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Performance Comparison — PTSD Framework', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim([0, 110]); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/Fig6_Model_Comparison.png', dpi=150)
plt.show()
print('✅ Fig6 saved')

In [ ]:
# ============================================================
# RESULTS SUMMARY TABLE (Paper Table I equivalent)
# ============================================================
print('\n' + '='*65)
print(f'{"Model":<18} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"F1 Score":>10}')
print('-'*65)
for name, res in results.items():
    print(f'{name:<18} {res["accuracy"]:>9.2f}% {res["precision"]:>9.2f}% '
          f'{res["recall"]:>7.2f}% {res["f1"]:>9.2f}%')
print('='*65)
print('Paper target: Accuracy > 95%')
print()
print('Output files:')
for f in os.listdir('outputs'):
    print(f'  outputs/{f}')

In [ ]:
# ============================================================
# DOWNLOAD ALL OUTPUTS (Colab only)
# ============================================================
try:
    from google.colab import files
    import zipfile
    with zipfile.ZipFile('PTSD_outputs.zip', 'w') as zf:
        for f in os.listdir('outputs'):
            zf.write(f'outputs/{f}', f)
    files.download('PTSD_outputs.zip')
    print('✅ Download started!')
except ImportError:
    print('ℹ️  Not running in Colab — files are in the outputs/ folder.')